In [ ]:
import gc
import json
import os.path as op
from pathlib import Path

import anndata as ad
import pandas as pd
import scanpy as sc
from sklearn.metrics import adjusted_rand_score

# Global environment settings
ad.settings.allow_write_nullable_strings = True
sc.settings.verbosity = 0

In [ ]:
def load_evidence(h5ad_path: str) -> ad.AnnData:
    """
    Loads an AnnData artifact from disk for final annotation mapping.

    Parameters
    ----------
    h5ad_path : str
        The absolute or relative path to the .h5ad file.

    Returns
    -------
    ad.AnnData
        The loaded expression matrix.
    """
    print(f"[INFO] Loading target matrix from {h5ad_path}")
    adata = sc.read_h5ad(h5ad_path)
    print(f"[INFO] Loaded dimensions: {adata.n_obs} cells x {adata.n_vars} genes")
    
    return adata

For macro_leiden: 
    **Macro Cluster 0**: Driven entirely by CD3D, IL32, and CD3E. The CD3 complex is the non-negotiable physical anchor for the T-cell receptor. This is the Lymphoid (T/NK) Lineage.
    **Macro Cluster 1**: Driven entirely by FCN1, FCER1G, and S100A9. These are the heavy lysozymes and receptors of the Myeloid Lineage.
    **Macro Cluster 2**: Driven entirely by CD79A, HLA-DQA1, and MS4A1 (CD20). The CD79 complex and CD20 are the physical anchors of the B-Cell Lineage.
-------
    The Major Clade (The Lymphocytes): Cluster 0 (T/NK cells) and Cluster 2 (B-cells) are linked together on one side of the primary bracket.
    The Isolated Outgroup (The Phagocytes): Cluster 1 (Myeloid cells) is pushed entirely to its own distant branch.
-------
    All blood cells start as a single Hematopoietic Stem Cell. That cell divides into two entirely distinct physical lineages: the Common Lymphoid Progenitor (CLP) and the Common Myeloid Progenitor (CMP).
-------
    Because Cluster 0 (T-cells) and Cluster 2 (B-cells) both physically evolved from the CLP, they share massive amounts of underlying structural and signaling mRNA (like basic lymphocyte physical architecture) that the Myeloid cells entirely lack. The math detected this shared evolutionary gravity and pulled 0 and 2 together, casting 1 out into the Myeloid void.
-------
    **The CD3 Void**: CD3D and CD3E are blazing in Cluster 0. They are pitch black in 1 and 2.
    **The CD20 Void**: MS4A1 (CD20) and CD79A are blazing in Cluster 2. They are pitch black in 0 and 1.
    **The Lysozyme Void**: FCN1 and S100A9 are blazing in Cluster 1. They are pitch black in 0 and 2.
-------
    **The Biological Reality**:
        This is the physical proof of Terminal Differentiation. When a stem cell commits to becoming a B-cell, it permanently supercoils the DNA for the T-cell receptor (CD3) and the Myeloid lysozymes (S100A9) around histones, physically locking those genes away so the ribosomes can never read them again. The matrixplot visualizes this permanent epigenetic shutdown as absolute darkness. If we ever see CD3D bleeding heavily into a B-cell macro-cluster, our physical sequencing droplet was contaminated.
-------
Hence the dictionary is:
    0: Lymphoid (T/NK) Lineage
    1: Myeloid Lineage
    2: B-Cell Lineage

For macro_leiden_0:
    For cluster 0:   
        - **The Physical Signal**: The algorithm extracted CCR7 as the absolute definining feature.
        - **The Biology**: CCR7 is a chemokine receptor whose sole physical purpose is to home a T-cell to the lymph nodes. A T-cell only requires this when it has not yet encountered an antigen and is waiting in the lymph node to be activated.
        - **Identity**: CD4+ Naive T-Cell.
    For cluster 1:
        - **The Physical Signal**: The CCR7 beacon has been extinguished. The cell has upregulated metabolic/structural transcripts like COTL1 and entirely lacks the cytotoxic weaponry of Clusters 2 and 3.
        We then checked an entire list of all genes available in var_names, and found that on log mean expression on log1 normal values, LTB was high than any other gene in this cluster. 
        - **The Biology**: LTB was the absolute anchor. Lymphotoxin Beta is the primary cytokine produced by CD4+ Helper T-cells. They use this molecule to talk to macrophages, direct B-cells to produce antibodies, and physically organize the structure of lymph nodes. Clusters 0 and 1 both showed massive, blazing expression of LTB (log-mean values of 3.0 to 4.0).
        It has left the lymph node (no CCR7), but it is not actively killing. It is patrolling. This transcriptomic silence combined with structural readiness is the definitive fingerprint of a Memory T-Cell. This massive metabolic expansion (COTL1) in Cluster 1, proves it had exited the quiet Naive state (0) and entered the active Memory/Effector state (1).
        - **Identity**: CD4+ Memory T-Cell.
    For cluster 2:
        - **The Physical Signal**: Massive upregulation of CCL5, NKG7, and GZMK (Granzyme K).
        - **The Biology**: The dominance of GZMK specifically marks a well-defined state of effector/cytotoxic T-cells.
        - **Identity**: CD8+ Cytotoxic T-Cell.
    For cluster 3:
        - **The Physical Signal**: Absolute dominance of GZMB (Granzyme B), GNLY (Granulysin), and FGFBP2.
        - **The Biology**: Compare Cluster 3 to Cluster 2. Cluster 3 has upgraded its weaponry from Granzyme K to Granzyme B (a vastly more potent protease), and turned on Granulysin to physically punch holes in bacterial membranes.
        - **Identity**: Natural Killer (NK) Cell.
    From the absence of the top genes from other clusters :
        FCN1, S100A9 (Myeloid), and CD79A, MS4A1 (B-cell) are pitch dark. No doublets exist.
        Notice that FCER1G and FCGR3A (CD16) are blazing yellow in cluster 3. NK cells physically require the CD16 receptor to perform Antibody-Dependent Cellular Cytotoxicity (ADCC).
    The Dendrogram Split: The Great Adaptive Divide
    Look at the primary brackets at the top of the plot. The dendrogram mathematically severs the matrix into two massive super-clades:
        Clade A: Sub-clusters 0 and 1.
        Clade B: Sub-clusters 2 and 3.
    The Ground-Up Biology:
        The algorithm detected that 0 and 1 are fundamentally the same type of machine, and 2 and 3 are fundamentally a different type of machine.
            0 and 1 are the Helper/Regulatory machines. They coordinate the immune response.
            2 and 3 are the Effector/Killer machines. They physically terminate compromised host cells.
    Cluster 3 (NK Cells) is ablaze with NKG7 and CCL5.
        - **The Physics of NKG7**: NKG7 is a structural protein required to build the physical bomb (the cytotoxic granule) that kills target cells. NK cells (Cluster 3) use it constantly. CD8+ T-cells (Cluster 2) also use it when they are fully activated to kill.
        - **The Physics of CCL5 (RANTES)**: This is a chemokine beacon. When an NK cell or a CD8+ T-cell is engaged in combat, it secretes CCL5 to attract Monocytes and other T-cells to the battlefield.
        - The shared expression is the absolute proof that Cluster 2 and Cluster 3 share the exact same physical kill mechanism Perforin/Granzyme exocytosis.
        - CCR7 and PLP2 are completely dark in the 2 and 3 clade.
            . This is not passive absence; it is active biological suppression
            CCR7 is the chemical lock that keeps a cell inside a Lymph Node.
            . If a CD8+ T-cell (Cluster 2) or an NK cell (Cluster 3) is circulating in the peripheral blood to hunt viruses, it must destroy its CCR7 receptors. 
            . If it expressed CCR7, it would be physically dragged back into the lymph node and trapped there, unable to fight the infection. The darkness on the dotplot is the visual proof of this physiological deployment.

Hence,the final dictionary for the macro_leiden_0 is:
    0: CD4+ Naive T-Cell
    1: CD4+ Memory T-Cell
    2: CD8+ Cytotoxic T-Cell
    3: CD16+ Natural Killer (NK) Cell

For macro_leiden_1:
    Sub-Cluster 1: 
        **The Physical Signal**: Absolute maximum expression of S100A12 and FOLR3.
        **The Biolog**: S100A12 is a powerful, pro-inflammatory antimicrobial protein. A cell transcribing this at maximum volume is actively patrolling the blood for bacterial targets.
        **Identity**: CD14+ Classical Monocyte.
-------
    Sub-Cluster 2: 
        **The Physical Signal**: Dominance of FCGR3A, RHOC, and MS4A7.
        **The Biology**: FCGR3A is the physical gene code for the CD16 receptor. The presence of this receptor mathematically forces this cluster into the non-classical state. These cells have down-regulated their violent inflammatory markers to become specialized vascular patrollers.
        **Identity**: CD16+ Non-Classical Monocyte.
-------
    Sub-Cluster 0: 
        **The Physical Signal**: The sole defining vector extracted by the algorithm is HLA-DQB1. 
        **The Biology**: Look at the MatrixPlot. Cluster 0 is dark purple for the inflammatory Monocyte markers, but blazing yellow for the MHC-Class II molecule. This cell has abandoned phagocytosis to dedicate its entire physical volume to antigen presentation. Plus moderate CD14 too.
        **Identity**: Dendritic Cell (DC).
-------       
    Evidence: matrixplot__absence_audit_macro_1.svg (Myeloid matrix forced to show T/B-cell genes)
        **-** Looking at the absolute lack of signal for CD3D, CD3E, CD79A, and MS4A1. They are practically black.
        **-** Note on biological overlap: HLA-DQA1 lights up in the Dendritic cell cluster, which is a perfect biological reality, as both B-cells and DCs are professional APCs. PLP2 lights up because it is a ubiquitous metabolic gene. The lineage boundaries are secure.
------- 
    The dendrogram links the three Myeloid states. It generally groups 0 (Dendritic Cells) and 2 (CD16+ Non-Classical Monocytes) together, or shows 1 (CD14+ Classical Monocytes) standing slightly apart with its massive inflammatory payload.
----
    The Ground-Up Biology:
        The Myeloid lineage operates on a continuum of maturation and specialization.
            **(Cluster 1)**: CD14+ Monocytes are freshly minted from the bone marrow. They are massive, blunt-force trauma cells loaded with S100A12 and S100A9 (antimicrobial alarms). They are the raw material.
            **(Cluster 2 & 0)**: As Monocytes spend time in the blood, some differentiate. They shut down their blunt-force S100A genes (which is why these genes go dark in Clusters 2 and 0). They upregulate sophisticated vascular patrolling hardware (FCGR3A / CD16) or differentiate completely into professional antigen presenters (Dendritic Cells, Cluster 0).
-------            
    If we look closely at the dotplot across the Myeloid matrix, you will see the shared story of the MHC-Class II complex (e.g., HLA-DQB1, HLA-DRA).
        **Cluster 0 (DCs)**: The dot is massive and blazing red. This cell's entire existence is dedicated to holding pieces of dead viruses in this HLA presentation window for T-cells to inspect.
        **Cluster 2 (CD16+ Monocytes)**: The dot for HLA-DQB1 is present, but smaller and paler. Why? Because as Monocytes mature into the non-classical state, they increase their ability to present antigens, bridging the gap between innate and adaptive immunity. They are not professionals like the DCs, but they share the hardware.
        **Cluster 1 (CD14+ Monocytes)**: The dot is at its minimum. They are too busy eating pathogens to worry about presenting them.
-------
Hence the Dictionary is:
    0: Dendritic Cell (DC)
    1: CD14+ Classical Monocyte
    2: CD16+ Non-Classical Monocyte

In [ ]:
def orc_annotation(
    dict_path: str, 
    annotation_manual_path: str, 
    ontology_cl_id_dict_path: str
) -> None:
    """
    Injects human-verified biological annotations and standard Cell Ontology (CL) 
    IDs into the localized Macro and Micro matrices. Calculates alignment 
    scores against automated reference models.

    Parameters
    ----------
    dict_path : str
        Path to the state dictionary (Orchestrator A or B).
    annotation_manual_path : str
        Path to the human-populated JSON ledger of biological identities.
    ontology_cl_id_dict_path : str
        Path to the JSON ledger mapping biological identities to CL IDs.

    Returns
    -------
    None
    """
    print("\n===========================================================")
    print(" INITIATING ANNOTATION INJECTION ENGINE")
    print("===========================================================")
    
    if not op.exists(dict_path):
        print(f"[ERROR] State dictionary missing at {dict_path}")
        return None
        
    with open(dict_path, 'r') as json_file:
        state_dict = json.load(json_file)
        
    if not op.exists(annotation_manual_path):
        print(f"[ERROR] Manual annotation ledger missing at {annotation_manual_path}")
        return None
        
    with open(annotation_manual_path, 'r') as json_file:
        annotation_manual = json.load(json_file)
        
    if not op.exists(ontology_cl_id_dict_path):
        print(f"[ERROR] Ontology dictionary missing at {ontology_cl_id_dict_path}")
        return None
        
    with open(ontology_cl_id_dict_path, 'r') as json_file:
        ontology_cl_id_dict_manual = json.load(json_file)

    macro_path_key, macro_leiden_key, micro_paths_key, micro_leiden_key = None, None, None, None
    
    for k in state_dict.keys():
        if 'split' in k or ("macro" in k and "file_path" in k and "dictionary" not in k):
            macro_path_key = k
        elif "macro" in k and "leiden_key" in k and "dictionary" not in k:
            macro_leiden_key = k
        elif "micro" in k and "file_path" in k and "dictionary" in k:
            micro_paths_key = k
        elif "micro" in k and "leiden_key" in k and "dictionary" in k:
            micro_leiden_key = k
            
    if not all([macro_path_key, macro_leiden_key, micro_paths_key, micro_leiden_key]):
        print("[ERROR] Matrix collapse. Could not autonomously identify core keys.")
        return None

    # MACRO INJECTION
    macro_payload = state_dict.get(macro_path_key)
    macro_path = None
    
    if isinstance(macro_payload, dict):
        for k in macro_payload.keys():
            if 'training' in k:
                macro_path = macro_payload.get(k)
                break
    else:
        macro_path = macro_payload
        
    macro_leiden = state_dict.get(macro_leiden_key)
    
    if macro_path and macro_leiden:
        if op.exists(macro_path):
            print(f"[MACRO] Mapping annotations for manifold: {macro_leiden}")
            adata = load_evidence(macro_path)
            
            # Vectorized assignment
            adata.obs['manual_labels'] = adata.obs[macro_leiden].map(annotation_manual[macro_leiden])
            adata.obs['human_CL_ID'] = adata.obs['manual_labels'].map(ontology_cl_id_dict_manual)
            
            if 'majority_voting' in adata.obs_keys():
                adata.obs['oracle_CL_ID'] = adata.obs['majority_voting'].map(ontology_cl_id_dict_manual)
                
                # Drop NaNs to allow strict ARI comparison
                valid_mask = adata.obs['human_CL_ID'].notna() & adata.obs['oracle_CL_ID'].notna()
                if valid_mask.sum() > 0:
                    ari_score = adjusted_rand_score(
                        adata.obs.loc[valid_mask, 'human_CL_ID'].astype(str),
                        adata.obs.loc[valid_mask, 'oracle_CL_ID'].astype(str)
                    )
                    adata.uns['Oracle_ARI_Score'] = float(ari_score)
                    print(f"  -> Oracle Alignment Score (ARI): {ari_score:.3f}")
                
            adata.write_h5ad(macro_path)
            del adata
            gc.collect()
        else:
            print(f"[ERROR] Macro file missing at {macro_path}")

    # MICRO INJECTION
    micro_paths_dict = state_dict.get(micro_paths_key, {})
    micro_leiden_dict = state_dict.get(micro_leiden_key, {})
    
    for leiden_dict_key, file_path in micro_paths_dict.items():
        print(f"\n[MICRO] Scanning topology: {leiden_dict_key}")
        
        if not op.exists(file_path):
            print(f"  -> [WARNING] Matrix missing at {file_path}. Bypassing.")
            continue
            
        active_leiden_col = micro_leiden_dict.get(leiden_dict_key)
        adata = load_evidence(file_path)
        
        if active_leiden_col is None:
            # Handle Terminal States via Parent Inheritance
            clean_key = leiden_dict_key.replace('_Terminal_State', '')
            parts = clean_key.split('_')
            cluster_id = parts[-1]
            parent_dict_key = '_'.join(parts[:-1])
            
            inherited_label = annotation_manual.get(parent_dict_key, {}).get(cluster_id)
            print(f"  -> Terminal State detected. Inheriting Parent Label: '{inherited_label}'")
            
            adata.obs['manual_labels'] = inherited_label
            adata.obs['human_CL_ID'] = adata.obs['manual_labels'].map(ontology_cl_id_dict_manual)
            
        else:
            # Handle Standard Active Micro States
            print(f"  -> Active State detected. Mapping via: '{active_leiden_col}'")
            adata.obs['manual_labels'] = adata.obs[active_leiden_col].map(
                annotation_manual[active_leiden_col]
            )
            adata.obs['human_CL_ID'] = adata.obs['manual_labels'].map(ontology_cl_id_dict_manual)

        # Oracle Alignment Check
        if 'majority_voting' in adata.obs_keys():
            adata.obs['oracle_CL_ID'] = adata.obs['majority_voting'].map(ontology_cl_id_dict_manual)
            
            valid_mask = adata.obs['human_CL_ID'].notna() & adata.obs['oracle_CL_ID'].notna()
            if valid_mask.sum() > 0:
                ari_score = adjusted_rand_score(
                    adata.obs.loc[valid_mask, 'human_CL_ID'].astype(str),
                    adata.obs.loc[valid_mask, 'oracle_CL_ID'].astype(str)
                )
                adata.uns['Oracle_ARI_Score'] = float(ari_score)
                print(f"  -> Oracle Alignment Score (ARI): {ari_score:.3f}")
                
        adata.write_h5ad(file_path)
        del adata
        gc.collect()

In [ ]:
def label_mapping_data_frame_all(dict_path: str, master_df_csv_path: str) -> None:
    """
    Extracts cell barcodes and injected labels from all isolated matrices 
    and concatenates them into a master CSV ledger. Resolves barcode collisions 
    by prioritizing the most recent execution state.

    Parameters
    ----------
    dict_path : str
        Path to the state dictionary containing matrix paths.
    master_df_csv_path : str
        Target output path for the central CSV ledger.

    Returns
    -------
    None
    """
    print("\n[AUDIT] Aggregating localized labels into Master Ledger...")
    
    if not op.exists(dict_path):
        print(f"[ERROR] State dictionary missing at {dict_path}")
        return None
        
    with open(dict_path, 'r') as json_file:
        state_dict = json.load(json_file)
        
    micro_paths_key = None
    for k in state_dict.keys():
        if "micro" in k and "file_path" in k and "dictionary" in k:
            micro_paths_key = k
            
    if not micro_paths_key:
        print("[ERROR] Matrix collapse. Could not identify micro paths.")
        return None
        
    micro_paths_dict = state_dict.get(micro_paths_key, {})
    new_ledgers = []
    
    for key_cluster_id, file_path in micro_paths_dict.items():
        if not op.exists(file_path):
            print(f"  -> [WARNING] Missing matrix at {file_path}. Skipping.")
            continue
            
        adata_micro = load_evidence(file_path)
        
        if 'manual_labels' in adata_micro.obs and 'human_CL_ID' in adata_micro.obs:
            clean_df = adata_micro.obs[['manual_labels', 'human_CL_ID']].copy()
            new_ledgers.append(clean_df)
        else:
            print(f"  -> [ERROR] Labels missing in {key_cluster_id}. Cannot extract.")
            
        del adata_micro
        gc.collect()
        
    if not new_ledgers:
        print("[ERROR] No valid ledgers extracted. Terminating aggregation.")
        return None
        
    current_run_ledger = pd.concat(new_ledgers)
    
    if op.exists(master_df_csv_path):
        print("  -> Existing ledger detected. Loading into RAM for integration...")
        master_df = pd.read_csv(master_df_csv_path, index_col=0)
        combined_ledger = pd.concat([master_df, current_run_ledger])
        
        duplicate_count = combined_ledger.index.duplicated().sum()
        if duplicate_count > 0:
            print(f"  -> [INFO] {duplicate_count} barcodes exist in historical ledger.")
            print("  -> Overwriting historical state with fresh RAM state...")
            combined_ledger = combined_ledger[~combined_ledger.index.duplicated(keep='last')]
    else:
        print("  -> No existing ledger found. Initiating genesis ledger...")
        combined_ledger = current_run_ledger
        
    final_duplicate_count = combined_ledger.index.duplicated().sum()
    if final_duplicate_count > 0:
        print("[ERROR] Unresolvable BARCODE COLLISION DETECTED.")
        return None
        
    Path(master_df_csv_path).parent.mkdir(parents=True, exist_ok=True)
    combined_ledger.to_csv(master_df_csv_path, index_label='cell_barcode')
    
    print(f"[SUCCESS] Ledger permanently sealed. Total unique cells: {len(combined_ledger)}")

In [ ]:
def main_artifact_labelling(main_h5ad_path: str, master_df_csv_path: str) -> str:
    """
    Ingests the master CSV ledger and maps the final biological identities 
    onto the raw, un-split global matrix. Exports the final ML-Ready artifact.

    Parameters
    ----------
    main_h5ad_path : str
        Path to the primary QC'd AnnData file.
    master_df_csv_path : str
        Path to the aggregated master CSV ledger.

    Returns
    -------
    str
        Path to the finalized ML-Ready .h5ad file.
    """
    print("\n===========================================================")
    print(" EXECUTING FINAL TENSOR RECOMBINATION")
    print("===========================================================")
    
    if not op.exists(main_h5ad_path):
        print(f"[ERROR] Global matrix missing at {main_h5ad_path}")
        return None
        
    if not op.exists(master_df_csv_path):
        print(f"[ERROR] Master ledger missing at {master_df_csv_path}")
        return None
        
    master_df = pd.read_csv(master_df_csv_path, index_col=0)
    adata_main = load_evidence(main_h5ad_path)
    
    # The Vectorized Recombination
    adata_main.obs['Final_ML_Ready_Label'] = adata_main.obs_names.map(master_df['manual_labels'])
    adata_main.obs['Final_ML_Ready_CL_ID'] = adata_main.obs_names.map(master_df['human_CL_ID'])
    
    void_count = adata_main.obs['Final_ML_Ready_Label'].isna().sum()
    if void_count > 0:
        print(f"  -> [WARNING] {void_count} cells did not map (QC voids or unannotated).")
        print("  -> Standardizing voids as 'Unknown/Filtered'.")
        adata_main.obs['Final_ML_Ready_Label'].fillna('Unknown/Filtered', inplace=True)
        adata_main.obs['Final_ML_Ready_CL_ID'].fillna('Unknown/Filtered', inplace=True)
        
    base_name, ext = op.splitext(main_h5ad_path)
    ml_ready_path = f"{base_name}_ML_Ready{ext}"
    
    print(f"[SEALED] Writing finalized Machine Learning Tensor to: {ml_ready_path}")
    adata_main.write_h5ad(ml_ready_path)
    
    del adata_main
    gc.collect()
    
    return ml_ready_path

In [ ]:
if __name__ == '__main__':
    # Absolute Definitions
    main_h5ad_path = '/Users/qgem/GitHub/PBMC3k-reproducible/notebooks/results/adata_main_qc.h5ad'
    dict_file_training_path = '/Users/qgem/GitHub/PBMC3k-reproducible/notebooks/results/Dictionary_of_returns_from_orch_A.json'
    dict_file_projected_path = '/Users/qgem/GitHub/PBMC3k-reproducible/notebooks/results/Dictionary_of_returns_from_orch_B.json'
    annotations_path = '/Users/qgem/GitHub/PBMC3k-reproducible/notebooks/results/annotation_manual.json'
    universal_ontology_id_path = '/Users/qgem/GitHub/PBMC3k-reproducible/notebooks/results/universal_ontology_id_dict.json'
    master_df_csv_path = '/Users/qgem/GitHub/PBMC3k-reproducible/notebooks/results/master_labels_df.csv'
    
    # 1. Map Training Set
    orc_annotation(dict_file_training_path, annotations_path, universal_ontology_id_path)
    label_mapping_data_frame_all(dict_file_training_path, master_df_csv_path)
    
    # 2. Map Projected Set
    orc_annotation(dict_file_projected_path, annotations_path, universal_ontology_id_path)
    label_mapping_data_frame_all(dict_file_projected_path, master_df_csv_path)
    
    # 3. Final Artifact Recombination
    main_artifact_labelling(main_h5ad_path, master_df_csv_path)
    
    print("\n[SUCCESS] PIPELINE COMPLETE. ML TENSOR IS READY FOR DEPLOYMENT.")